# 📊 Tahap 3: Evaluasi & Inferensi Model YOLOv8

**Proyek Replikasi Penelitian YOLOv8**  
Dataset: Recyclable Waste (Roboflow)

### Deskripsi Tahap Ini:
Setelah model selesai dilatih, kita perlu mengevaluasi performanya secara detail. Notebook ini mencakup:
1. Memuat bobot model terbaik hasil training (`best.pt`).
2. Menguji model pada dataset test untuk mendapatkan nilai Precision, Recall, dan mAP secara objektif.
3. Menampilkan grafik Confusion Matrix dan grafik Loss hasil training.
4. Melakukan prediksi pada citra eksternal di folder `citra_uji_eksternal` dan memvisualisasikan hasilnya secara langsung.

### 🛠️ 1. Setup Environment dan Import Library
Muat dependensi dan tentukan path ke file bobot model terbaik hasil training Anda.

In [ ]:
import os
import glob
import cv2
import matplotlib.pyplot as plt
from IPython.display import Image, display
from ultralytics import YOLO

# Tentukan path root proyek secara dinamis
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))
data_yaml_path = os.path.join(project_root, "data.yaml")

# Cari bobot terbaik dari folder training yang ada
weights_paths = [
    os.path.join(project_root, "runs", "detect", "train-yolov8n", "weights", "best.pt"),
    os.path.join(project_root, "runs", "detect", "train-yolov8s", "weights", "best.pt"),
    os.path.join(project_root, "runs", "detect", "train-3", "weights", "best.pt"),
    os.path.join(project_root, "yolov8n.pt")  # Fallback
]

best_weights = None
for path in weights_paths:
    if os.path.exists(path):
        best_weights = path
        break

if best_weights:
    print(f"✅ Bobot model terbaik ditemukan di: {best_weights}")
else:
    print("❌ Error: File best.pt tidak ditemukan. Jalankan training.ipynb terlebih dahulu!")

### 📈 2. Evaluasi Model pada Dataset Test
Jalankan cell di bawah untuk menghitung metrik performa secara otomatis pada dataset test (737 gambar).

In [ ]:
if best_weights and os.path.exists(data_yaml_path):
    # Muat model
    model = YOLO(best_weights)
    
    # Evaluasi pada dataset pengujian
    print("⏳ Mengevaluasi model...")
    metrics = model.val(data=data_yaml_path, split='test', plots=True)
    
    print("\n" + "="*40)
    print("🏆 METRIK AKURASI MODEL (TEST SET):")
    print("="*40)
    print(f"Precision (Presisi) : {metrics.box.mp:.4f} ({metrics.box.mp*100:.1f}%)")
    print(f"Recall (Daya Ingat) : {metrics.box.mr:.4f} ({metrics.box.mr*100:.1f}%)")
    print(f"mAP50                : {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)")
    print(f"mAP50-95             : {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)")
    print("="*40)
else:
    print("❌ Konfigurasi atau file bobot model tidak lengkap.")

### 📊 3. Visualisasi Hasil Pelatihan
Ultralytics menghasilkan visualisasi grafik secara otomatis selama proses pelatihan. Mari kita tampilkan grafik **Confusion Matrix** dan kurva performa **results.png** hasil training.

In [ ]:
# Tentukan folder training
train_folder = os.path.dirname(os.path.dirname(best_weights)) if best_weights else None

if train_folder and os.path.exists(train_folder):
    results_img = os.path.join(train_folder, "results.png")
    cm_img = os.path.join(train_folder, "confusion_matrix.png")
    
    print("📈 MENAMPILKAN KURVA HASIL TRAINING (Loss, Precision, Recall, mAP):")
    if os.path.exists(results_img):
        display(Image(filename=results_img, width=800))
    else:
        print("results.png tidak ditemukan.")
        
    print("\n📊 MENAMPILKAN CONFUSION MATRIX:")
    if os.path.exists(cm_img):
        display(Image(filename=cm_img, width=600))
    else:
        print("confusion_matrix.png tidak ditemukan.")
else:
    print("Folder training tidak terdeteksi.")

### 🔮 4. Uji Inferensi (Prediksi Citra Uji Eksternal)
Sekarang mari kita uji model pada citra uji di luar dataset (`citra_uji_eksternal`) dan tampilkan hasilnya secara visual.

In [ ]:
test_images_dir = os.path.join(project_root, "citra_uji_eksternal")
image_files = glob.glob(os.path.join(test_images_dir, "*.jpg")) + \
              glob.glob(os.path.join(test_images_dir, "*.jpeg")) + \
              glob.glob(os.path.join(test_images_dir, "*.png")) + \
              glob.glob(os.path.join(test_images_dir, "*.webp"))

if best_weights and image_files:
    model = YOLO(best_weights)
    
    # Lakukan prediksi
    print(f"Menjalankan inferensi pada {len(image_files)} citra eksternal...")
    results = model.predict(source=test_images_dir, save=True, conf=0.25)
    
    # Menampilkan hasil gambar prediksi
    # Ultralytics menyimpan hasil di runs/detect/predict (atau predict2, predict3, dst.)
    # Cari folder predict terbaru
    predict_folders = glob.glob(os.path.join(project_root, "runs", "detect", "predict*"))
    if predict_folders:
        latest_predict_folder = max(predict_folders, key=os.path.getmtime)
        print(f"\n🖼️ Citra terprediksi disimpan di: {latest_predict_folder}")
        
        # Tampilkan setiap gambar hasil prediksi
        predicted_images = glob.glob(os.path.join(latest_predict_folder, "*"))
        for p_img in predicted_images:
            if p_img.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                filename = os.path.basename(p_img)
                print(f"Hasil Prediksi: {filename}")
                display(Image(filename=p_img, width=500))
                print("-" * 50)
else:
    print("❌ Tidak ada citra uji eksternal atau model tidak siap.")

### 🏁 Kesimpulan Tahap 3
Model YOLOv8 berhasil diuji dengan performa yang memuaskan. Model mampu mendeteksi berbagai jenis objek sampah daur ulang (kaca, kertas, logam, plastik) pada citra luar dataset secara akurat.